# RAG-Adapter research notebook
See `docs/notebooks.md` before running individual experiment sections. Outputs are cleared and local paths use the configuration cell below.


In [ ]:
from __future__ import annotations
import os
from pathlib import Path
RELEASE_ROOT = Path.cwd().resolve()
if RELEASE_ROOT.name == "notebooks":
    RELEASE_ROOT = RELEASE_ROOT.parent
DATA_ROOT = str(Path(os.environ.get("RAG_DATA_ROOT", RELEASE_ROOT / "data_local")).expanduser().resolve())
WORK_ROOT = str(Path(os.environ.get("RAG_WORK_ROOT", RELEASE_ROOT / "runs")).expanduser().resolve())
BGE_MODEL = os.environ.get("RAG_BGE_MODEL", "BAAI/bge-m3")
Path(WORK_ROOT).mkdir(parents=True, exist_ok=True)


# 加载python库

In [ ]:
from collections import defaultdict
import pyarrow.parquet as pq
from openai import OpenAI
import os 
import json
import re
import numpy as np
from difflib import SequenceMatcher
import re
import numpy as np


# 加在openai客户端

In [ ]:
from openai import OpenAI
client = OpenAI()  # Reads OPENAI_API_KEY and optional OPENAI_BASE_URL.

def call_gpt_with_messages(messages, model_name):
    if model_name == "gpt_4o":
        model = "gpt-4o-2024-08-06"
    if model_name == "gpt_4_turbo":
        model = "gpt-4-turbo-2024-04-09"
    if model_name == "gemini-2.0-flash":
        model = "gemini-2.0-flash"
    response = client.chat.completions.create(
        model=model, 
        messages=messages,
        max_tokens=int(os.environ.get("RAG_JUDGE_MAX_TOKENS", "4096")),
        temperature=0.7,
        )
    return response


# 加载数据集问题

## Video-MME

In [ ]:
data = pq.ParquetFile(f"{DATA_ROOT}/dataset/Video-MME/test-00000-of-00001.parquet")
table = data.read()

questions_video_mme = defaultdict()

video_ids = []
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Video-MME/frames"):
    if len(ds) == 0:
        video_id = root.split("/")[-1]
        video_ids.append(video_id)


for i in range(data.num_row_groups):
    row_group = data.read_row_group(i)
    row_group = row_group.to_pandas()
    for idx, row in row_group.iterrows():
        if row.videoID in video_ids:
            # print(row.question_id)
            if row.videoID not in questions_video_mme:
                questions_video_mme[row.videoID] = {}
            if row.question_id not in questions_video_mme[row.videoID]:
                questions_video_mme[row.videoID][row.question_id] = {}
            questions_video_mme[row.videoID][row.question_id]["question"] = row.question
            questions_video_mme[row.videoID][row.question_id]["options"] = row.options
            questions_video_mme[row.videoID][row.question_id]["answer"] = row.answer
            questions_video_mme[row.videoID][row.question_id]["duration"] = row.duration
            questions_video_mme[row.videoID][row.question_id]["domain"] = row.domain


## MLVU

In [ ]:
tasks = ['1_plotQA', '2_needle', '3_ego', '4_count', '5_order', '6_anomaly_reco', '7_topic_reasoning']
# '8_sub_scene', '9_summary'
options = ['A', 'B', 'C', 'D']
questions_mlvu = defaultdict(dict)
video_ids = []
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/MLVU/frames"):
    if len(ds) == 0:
        video_id = root.split("/")[-1]
        video_ids.append(video_id)

for task in tasks:
    file = os.path.join(f"{DATA_ROOT}/dataset/MLVU", f"MLVU_json_{task}.json")
    with open(file, 'r', encoding='utf-8') as f:
        qs = json.load(f)
    for q in qs:
        video_id = q["video"].split(".")
        video_id = video_id[0]
        if video_id not in video_ids:
            continue
        
        question_id = q["question"][:40]
        if question_id not in questions_mlvu[video_id]:
            questions_mlvu[video_id][question_id] = dict()
        if "candidates" in q:
            questions_mlvu[video_id][question_id]["candidates"] = q["candidates"]
            for i, c in enumerate(q["candidates"]):
                if c == q["answer"]:
                    questions_mlvu[video_id][question_id]["answer"] = options[i]
            if questions_mlvu[video_id][question_id]["answer"] not in options:
                print(f"video: {video_id}, question: {question_id}, answer: {q['answer']}")
        else:
            questions_mlvu[video_id][question_id]["answer"] = q["answer"]
        if "scoring_points" in q:
            questions_mlvu[video_id][question_id]["scoring_points"] = q["scoring_points"]
        questions_mlvu[video_id][question_id]["question"] = q["question"]
        questions_mlvu[video_id][question_id]["duration"] = q["duration"]
        questions_mlvu[video_id][question_id]["question_type"] = task


## Perception_Test

In [ ]:
with open(f"{DATA_ROOT}/dataset/Perception_Test/mc_question_train.json", "r") as json_file:
    mc_question = json.load(json_file)

questions_perception_test = defaultdict()
alts = ["A", "B", "C"]
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/Perception_Test/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_id = f.split(".mp4")[0]
            if video_id not in questions_perception_test:
                questions_perception_test[video_id] = defaultdict(list)
            for q in mc_question[video_id]["mc_question"]:
                questions_perception_test[video_id]["id"].append(q["id"])
                questions_perception_test[video_id]["question"].append(q["question"])
                index_options = []
                for i, opt in enumerate(q["options"]):
                    index_options.append(f"{alts[i]}. {opt}")
                questions_perception_test[video_id]["options"].append(index_options)
                questions_perception_test[video_id]["answer_id"].append(alts[int(q["answer_id"])])


## EgoSchema

In [ ]:
with open(f"{DATA_ROOT}/dataset/EgoSchema/questions.json", "r") as q_file:
    mc_questions = json.load(q_file)

with open(f"{DATA_ROOT}/dataset/EgoSchema/subset_answers.json", "r") as a_file:
    mc_answers = json.load(a_file)

questions_egoschema = defaultdict()
alts = ["A", "B", "C", "D", "E"]
for root, ds, fs in os.walk(f"{DATA_ROOT}/dataset/EgoSchema/sampled_videos"):
    if len(ds) == 0:
        for f in fs:
            video_id = f.split(".mp4")[0]
            if video_id not in questions_egoschema:
                questions_egoschema[video_id] = defaultdict(list)
            for video in mc_questions:

                if video["q_uid"] == video_id:
                    questions_egoschema[video_id]["question"].append(video["question"])

                    index_options = []
                    index_options.append(f"A. {video['option 0']}")
                    index_options.append(f"B. {video['option 1']}")
                    index_options.append(f"C. {video['option 2']}")
                    index_options.append(f"D. {video['option 3']}")
                    index_options.append(f"E. {video['option 4']}")

                    questions_egoschema[video_id]["options"].append(index_options)
                    questions_egoschema[video_id]["answer_id"].append(alts[int(mc_answers[video_id])])


# 测试Video-MME的model准确率

In [ ]:
# model = "moviechat_10"
# model = "llamavid_10"
# model = "timechat_5"
# model = "chat-univi_10"

# model = "videoxl_10"
# model = "videoxl_100"
# model = "llava_video_10"
# model = "llava_video_32"

# rag_type = "gc_rag_uni_srt"
# rag_type = "gc_rag_gc_srt"
# rag_type = "sc_rag"
# rag_type = "gc_rag"
# rag_type = "two_stage"
# rag_type = "gc_rag_search_by_only_captions"
# rag_type = "gc_rag_search_by_only_frames"
# rag_type = "gc_rag_search_without_dual_ranker"
# rag_type = "sc_rag_construct_batches"
# rag_type = "no_ft_rag"
# rag_type = "no_rag"

rag_type = "ground_truth_rag"
# rag_type = "internvit6b_rag"
# rag_type = "sevila_rag"
# rag_type = "internvl14b_rag"

def acc():
    alt = ["A", "B", "C", "D"]
    if os.path.exists("error_format.txt"):
        os.remove("error_format.txt")
    for root, ds, fs in os.walk(f"{DATA_ROOT}/results/Video-MME/{model}/{rag_type}"):
        if len(ds) != 0:
            for d in ds:
                video_id = d
                preds = os.listdir(os.path.join(root, d))
                if len(preds) != 3:
                    print(f"{video_id} does not contain 3 answer files!")
                for pred in preds:
                    question_id = pred.split("_pred.txt")[0]
                    file_path = os.path.join(root, video_id, pred)
                    with open(file_path, 'r', encoding='utf-8') as pred_file:
                        text = pred_file.read()
                    if video_id not in questions_video_mme:
                        print(f"Video ID : {video_id} does not exist!")
                    patterns = ["([ABCD])","^([ABCD])", "([ABCD])\.", "The correct answer is ([ABCD])", "The correct answer is: ([ABCD])", "The best answer for this question is ([ABCD])",
                                "The best answer is ([ABCD])", "is ([ABCD])", "The correct answer is \"([ABCD])", "The best answer is \"([ABCD])",
                                "The answer is \"([ABCD])", "The correct answer is option ([ABCD])",
                                "question is: ([ABCD])", "best answer is: ([ABCD])", "question is \"([ABCD])",
                                "option is \"([ABCD])", "answer is \(([ABCD])", "\(([ABCD])", "\[([ABCD])"]
                    for p in patterns:
                        ans = re.findall(p, text)
                        if len(ans) != 0:
                            break
                    if len(ans) == 0:
                        with open("error_format.txt", 'a', encoding='utf-8') as f:
                            f.write(text+"\n")
                        # print(f"{video_id}-{pred}'s format is not correct!")
                        # sim = []
                        # for i in answers[video_id][question_id]["options"]:
                        #     sim.append(SequenceMatcher(None, i, text).ratio())
                        # answers[video_id][question_id]["pred"] = alt[sim.index(max(sim))]
                        questions_video_mme[video_id][question_id]["pred"] = alt[np.random.randint(0, 4)]
                        # answers[video_id][question_id]["pred"] = "Error"                       
                    else:    
                        questions_video_mme[video_id][question_id]["pred"] = ans[0]

    domain = ["Knowledge", "Film & Television", "Sports Competition", "Artistic Performance", "Life Record", "Multilingual"]
    duration = ["short", "medium", "long"]
    correct_domain = [0] * len(domain)
    total_domain = [0] * len(domain)
    correct_duration = [0] * len(duration)
    total_duration = [0] * len(duration)
    for video_id in questions_video_mme.keys():
        for question_id in questions_video_mme[video_id].keys():
            if questions_video_mme[video_id][question_id]["answer"] == questions_video_mme[video_id][question_id]["pred"]:
                domain_index = domain.index(questions_video_mme[video_id][question_id]["domain"])
                correct_domain[domain_index] += 1
                total_domain[domain_index] += 1

                duration_index = duration.index(questions_video_mme[video_id][question_id]["duration"])
                correct_duration[duration_index] += 1
                total_duration[duration_index] += 1
            else:
                domain_index = domain.index(questions_video_mme[video_id][question_id]["domain"])
                total_domain[domain_index] += 1

                duration_index = duration.index(questions_video_mme[video_id][question_id]["duration"])
                total_duration[duration_index] += 1
                
    # for i, d in enumerate(domain):
    #     print(f"{d}准确率: {correct_domain[i]/total_domain[i]}")
    # print(f"平均准确率: {sum(correct_domain)/sum(total_domain)}")

    for i, d in enumerate(duration):
        print(f"{d}准确率: {correct_duration[i]/total_duration[i]}")
    print(f"平均准确率: {sum(correct_duration)/sum(total_duration)}")
acc()               


# 测试Perception Test的准确率

In [ ]:
# model = "moviechat_10"
# model = "llamavid_10"
# model = "timechat_10"
# model = "gpt_4o_10"
model = "chat-univi_10"
# rag_type = "gc_rag_uni_srt"
# rag_type = "gc_rag_gc_srt"
# rag_type = "sc_rag"
# rag_type = "gc_rag"
# rag_type = "gc_rag_search_by_only_captions"
# rag_type = "gc_rag_search_by_only_frames"
# rag_type = "gc_rag_search_without_dual_ranker"
# rag_type = "sc_rag_construct_batches"
# rag_type = "no_ft_rag"
# rag_type = "no_rag"

def acc():
    alt = ["A", "B", "C"]
    if os.path.exists("error_format.txt"):
        os.remove("error_format.txt")
    for root, ds, fs in os.walk(f"{DATA_ROOT}/results/Perception_Test/{model}/{rag_type}"):
        if len(ds) != 0:
            for d in ds:
                video_id = d
                preds = os.listdir(os.path.join(root, d))
                for pred in preds:
                    question_id = pred.split("_pred.txt")[0]
                    file_path = os.path.join(root, video_id, pred)
                    with open(file_path, 'r', encoding='utf-8') as pred_file:
                        text = pred_file.read()
                    patterns = ["^([ABC])", "([ABC])\.", "The correct answer is ([ABC])", "The correct answer is: ([ABC])", "The best answer for this question is ([ABC])",
                                "The best answer is ([ABC])", "is ([ABC])", "The correct answer is \"([ABC])", "The best answer is \"([ABC])",
                                "The answer is \"([ABC])", "The correct answer is option ([ABC])",
                                "question is: ([ABC])", "best answer is: ([ABC])", "question is \"([ABC])",
                                "option is \"([ABC])", "answer is \(([ABC])", "\(([ABC])", "\[([ABC])", "answer is option ([ABC])"]
                    for p in patterns:
                        ans = re.findall(p, text)
                        if len(ans) != 0:
                            break
                    if len(ans) == 0:
                        with open("error_format.txt", 'a', encoding='utf-8') as f:
                            f.write(text+"\n")
                        # print(f"{video_id}-{pred}'s format is not correct!")
                        # sim = []
                        # for i in answers[video_id]["options"]:
                        #     sim.append(SequenceMatcher(None, i, text).ratio())
                        # answers[video_id][question_id]["pred"] = alt[sim.index(max(sim))]
                        questions_perception_test[video_id]["pred"].append(alt[np.random.randint(0, 3)])
                        # answers[video_id][question_id]["pred"] = "Error"                       
                    else:    
                        questions_perception_test[video_id]["pred"].append(ans[0])
                        # answers[video_id][question_id]["pred"] = ans[0]

    correct=0
    total=0
    for video_id in questions_perception_test.keys():
        for i, answer in enumerate(questions_perception_test[video_id]["pred"]):
            if questions_perception_test[video_id]["answer_id"][i] == answer:
                correct += 1
                total += 1
            else:
                total+= 1
    print(f"准确率: {correct/total}")
acc()               


# 测试Egochema的准确率

In [ ]:
# model = "moviechat_10"
# model = "llamavid_10"
# model = "timechat_10"
model = "chat-univi_10"
# rag_type = "gc_rag_uni_srt"
# rag_type = "gc_rag_gc_srt"
# rag_type = "sc_rag"
# rag_type = "gc_rag"
# rag_type = "gc_rag_search_by_only_captions"
# rag_type = "gc_rag_search_by_only_frames"
# rag_type = "gc_rag_search_without_dual_ranker"
# rag_type = "sc_rag_construct_batches"
# rag_type = "no_ft_rag"
rag_type = "no_rag"

def acc():
    alt = ["A", "B", "C", "D", "E"]
    if os.path.exists("error_format.txt"):
        os.remove("error_format.txt")
    for root, ds, fs in os.walk(f"{DATA_ROOT}/results/Egoschema/{model}/{rag_type}"):
        if len(ds) != 0:
            for d in ds:
                video_id = d
                preds = os.listdir(os.path.join(root, d))
                for pred in preds:
                    question_id = pred.split("_pred.txt")[0]
                    file_path = os.path.join(root, video_id, pred)
                    with open(file_path, 'r', encoding='utf-8') as pred_file:
                        text = pred_file.read()
                    patterns = ["^([ABCDE])", "([ABCDE])\.", "The correct answer is ([ABCDE])", "The correct answer is: ([ABCDE])", "The best answer for this question is ([ABCDE])",
                                "The best answer is ([ABCDE])", "is ([ABCDE])", "The correct answer is \"([ABCDE])", "The best answer is \"([ABCDE])",
                                "The answer is \"([ABCDE])", "The correct answer is option ([ABCDE])",
                                "question is: ([ABCDE])", "best answer is: ([ABCDE])", "question is \"([ABCDE])",
                                "option is \"([ABCDE])", "answer is \(([ABCDE])", "\(([ABCDE])", "\[([ABCDE])",
                                "I'd go with ([ABCDE])", "I agree with ([ABCDE])", "answer is option ([ABCDE])"]
                    for p in patterns:
                        ans = re.findall(p, text)
                        if len(ans) != 0:
                            break
                    if len(ans) == 0:
                        with open("error_format.txt", 'a', encoding='utf-8') as f:
                            f.write(text+"\n\n")
                        # print(f"{video_id}-{pred}'s format is not correct!")
                        # sim = []
                        # for i in answers[video_id]["options"]:
                        #     sim.append(SequenceMatcher(None, i, text).ratio())
                        # answers[video_id][question_id]["pred"] = alt[sim.index(max(sim))]
                        questions_egoschema[video_id]["pred"].append(alt[np.random.randint(0, 3)])
                        # answers[video_id][question_id]["pred"] = "Error"                       
                    else:    
                        questions_egoschema[video_id]["pred"].append(ans[0])
                        # answers[video_id][question_id]["pred"] = ans[0]

    correct=0
    total=0
    for video_id in questions_egoschema.keys():
        for i, answer in enumerate(questions_egoschema[video_id]["pred"]):
            if questions_egoschema[video_id]["answer_id"][i] == answer:
                correct += 1
                total += 1
            else:
                total+= 1
    print(f"准确率: {correct/total}")
acc()               


# 测试MLVU的model准确率

In [ ]:
model = "gpt_4o_20"
# rag_type = "gc_rag"
rag_type = "no_rag"

# multi-choice
def acc():
    alt = ["A", "B", "C", "D"]
    if os.path.exists("error_format.txt"):
        os.remove("error_format.txt")
    for root, ds, fs in os.walk(f"{DATA_ROOT}/results/MLVU/{model}/{rag_type}"):
        if len(ds) != 0:
            for video_id in ds:
                preds = os.listdir(os.path.join(root, video_id))
                for pred in preds:
                    if pred.endswith("_pred.txt"):
                        question_id = pred.split("_pred.txt")[0]
                    else:
                        continue
                    # print(f"{video_id}-{question_id}")
                    if questions_mlvu[video_id][question_id]["question_type"] in ['8_sub_scene', '9_summary']:
                        continue

                    file_path = os.path.join(root, video_id, pred)
                    with open(file_path, 'r', encoding='utf-8') as pred_file:
                        text = pred_file.read()
                    if video_id not in questions_mlvu.keys():
                        print(f"Video ID : {video_id} does not exist!")

                    patterns = ["^([ABCD])", "([ABCD])\.", "The correct answer is ([ABCD])", "The correct answer is: ([ABCD])", "The best answer for this question is ([ABCD])",
                                "The best answer is ([ABCD])", "is ([ABCD])", "The correct answer is \"([ABCD])", "The best answer is \"([ABCD])",
                                "The answer is \"([ABCD])", "The correct answer is option ([ABCD])",
                                "question is: ([ABCD])", "best answer is: ([ABCD])", "question is \"([ABCD])",
                                "option is \"([ABCD])", "answer is \(([ABCD])", "best option is \(([ABCD])"]
                    for p in patterns:
                        ans = re.findall(p, text)
                        if len(ans) != 0:
                            break
                    if len(ans) == 0:
                        with open("error_format.txt", 'a', encoding='utf-8') as f:
                            f.write(text+"\n")
                        # print(f"{video_id}-{pred}'s format is not correct!")
                        # sim = []
                        # for i in questions[video_id][question_id]["candidates"]:
                        #     sim.append(SequenceMatcher(None, i, text).ratio())
                        # questions[video_id][question_id]["pred"] = alt[sim.index(max(sim))]
                        questions_mlvu[video_id][question_id]["pred"] = alt[np.random.randint(0, 4)]
                        # questions[video_id][question_id]["pred"] = "Error"                       
                    else:    
                        questions_mlvu[video_id][question_id]["pred"] = ans[0]

    tasks = ['1_plotQA', '2_needle', '3_ego', '4_count', '5_order', '6_anomaly_reco', '7_topic_reasoning']
    correct = [0] * len(tasks)
    total = [0] * len(tasks)
    for video_id in questions_mlvu.keys():
        for question_id in questions_mlvu[video_id].keys():
            if questions_mlvu[video_id][question_id]["question_type"] in ['8_sub_scene', '9_summary']:
                continue
            if questions_mlvu[video_id][question_id]["answer"] == questions_mlvu[video_id][question_id]["pred"]:
                task_index = tasks.index(questions_mlvu[video_id][question_id]["question_type"])
                correct[task_index] += 1
                total[task_index] += 1
            else:
                task_index = tasks.index(questions_mlvu[video_id][question_id]["question_type"])
                total[task_index] += 1
                
    for i, d in enumerate(tasks):
        print(f"{d}准确率: {correct[i]/total[i]}")
    print(f"平均准确率: {sum(correct)/sum(total)}")
acc()               


# 生成MLVU的model生成准确率

In [ ]:
model = "gpt_4o_20"
# rag_type = "gc_rag"
rag_type = "no_rag"

# open-ended
def sub_scene_score():
    for root, ds, fs in os.walk(f"{DATA_ROOT}/results/MLVU/{model}/{rag_type}"):
        if len(ds) != 0:
            for video_id in ds:
                preds = os.listdir(os.path.join(root, video_id))
                for pred in preds:
                    if pred.endswith("_pred.txt"):
                        question_id = pred.split("_pred.txt")[0]
                    else:
                        continue
                    if questions_mlvu[video_id][question_id]["question_type"] not in ['8_sub_scene']:
                        continue

                    question = questions_mlvu[video_id][question_id]["question"]
                    score_points = questions_mlvu[video_id][question_id]["scoring_points"]
                    file_path = os.path.join(root, video_id, pred)
                    with open(file_path, 'r', encoding='utf-8') as pred_file:
                        prediction = pred_file.read()
                    if video_id not in questions_mlvu.keys():
                        print(f"Video ID : {video_id} does not exist!")
                    

                    # evaluation prompt for sub-scene captioning task
                    query = f"""##TASK DESCRIPTION: You are required to evaluate a respondent's answer based on a provided question, some scoring points, and the respondent's answer. You should provide two scores. The first is the accuracy score, which should range from 1 to 5. The second is the relevance score, which should also range from 1 to 5. Below are the criteria for each scoring category.

##QUESTION: 
{question}

##SCORING POINTS:
{score_points}

##RESPONDENT'S ANSWER:
{prediction}

##ACCURACY Scoring Criteria: 
Evaluate the respondent's answer against specific scoring points as follows: 
Score 1: The response completely misses the scoring point. 
Score 3: The response mentions content related to the scoring point but is not entirely correct. 
Score 5: The response accurately addresses the scoring point. 
Calculate the average score across all scoring points to determine the final accuracy score. 

##RELEVANCE Scoring Criteria: Assess how the respondent's answer relates to the original question:
Score 1: The response is completely off-topic from the question. 
Score 2: The response is partially related to the question but contains a significant amount of irrelevant content. 
Score 3: The response primarily addresses the question, but the respondent seems uncertain about their own answer. 
Score 4: The response mostly addresses the question and the respondent appears confident in their answer. 
Score 5: The response is fully focused on addressing the question with no irrelevant content and demonstrates complete certainty. 

##INSTRUCTION: 
1. Evaluate ACCURACY: First, assess and score each scoring point based on the respondent's answer. Calculate the average of these scores to establish the final accuracy score. Provide a detailed rationale before assigning your score. 
2. Evaluate RELEVANCE: Assess the relevance of the respondent’s answer to the question. Note that when evaluating relevance, the correctness of the answer is not considered; focus solely on how relevant the answer is to the question. Provide a comprehensive rationale before assigning your score.
3. Output Scores in JSON Format: Present the scores in JSON format as follows:
{{
    "accuracy": 3,
    "relevance": 4
}}"""
                    
                    messages = [
                        {"role": "user", 
                        "content": [
                                {
                                    "type": "text",
                                    "text": query
                                }
                            ]
                        },
                    ]
                    output = ""
                    response = call_gpt_with_messages(messages, "gpt_4o")
                    output += response.choices[0].message.content

                    # print("query: ", query)
                    # print("answer: ", output + "\n")
                    save_file = os.path.join(f"{DATA_ROOT}/results/MLVU", model, rag_type, video_id, f"{question_id}_sub_score.txt")
                    with open(save_file, 'w') as f:
                        f.write(output)

def sum_score():
    for root, ds, fs in os.walk(f"{DATA_ROOT}/results/MLVU/{model}/{rag_type}"):
        if len(ds) != 0:
            for video_id in ds:
                preds = os.listdir(os.path.join(root, video_id))
                for pred in preds:
                    if pred.endswith("_pred.txt"):
                        question_id = pred.split("_pred.txt")[0]
                    else:
                        continue
                    if questions_mlvu[video_id][question_id]["question_type"] not in ['9_summary']:
                        continue

                    answer = questions_mlvu[video_id][question_id]["answer"]
                    file_path = os.path.join(root, video_id, pred)
                    with open(file_path, 'r', encoding='utf-8') as pred_file:
                        prediction = pred_file.read()
                    if video_id not in questions_mlvu.keys():
                        print(f"Video ID : {video_id} does not exist!")
                    

                    # evaluation prompt for sub-scene captioning task
                    query = f"""##TASK DESCRIPTION: You are required to evaluate the performance of the respondent in the video summarization task based on the standard answer and the respondent's answer. You should provide two scores. The first is the COMPLETENESS score, which should range from 1 to 5. The second is the RELIABILITY score, which should also range from 1 to 5. Below are the criteria for each scoring category:

##STANDARD ANSWER: 
{answer}

##RESPONDENT'S ANSWER:
{prediction}

##COMPLETENESS Scoring Criteria: 
The completeness score focuses on whether the summary covers all key points and main information from the video. 
Score 1: The summary hardly covers any of the main content or key points of the video. 
Score 2: The summary covers some of the main content and key points but misses many. 
Score 3: The summary covers most of the main content and key points. 
Score 4: The summary is very comprehensive, covering most to nearly all of the main content and key points. 
Score 5: The summary completely covers all the main content and key points of the video.

##CORRECTNESS Scoring Criteria: 
The correctness score evaluates the correctness and clarity of the video summary. It checks for factual errors, misleading statements, and contradictions with the video content. If the respondent's answer includes details that are not present in the standard answer, as long as these details do not conflict with the correct answer and are reasonable, points should not be deducted. 
Score 1: Contains multiple factual errors and contradictions; presentation is confusing. 
Score 2: Includes several errors and some contradictions; needs clearer presentation. 
Score 3: Generally accurate with minor errors; minimal contradictions; reasonably clear presentation. 
Score 4: Very accurate with negligible inaccuracies; no contradictions; clear and fluent presentation. 
Score 5: Completely accurate with no errors or contradictions; presentation is clear and easy to understand.

##INSTRUCTION: 
1. Evaluate COMPLETENESS: First, analyze the respondent's answer according to the scoring criteria, then provide an integer score between 1 and 5 based on sufficient evidence. 
2. Evaluate CORRECTNESS : First, analyze the respondent's answer according to the scoring criteria, then provide an integer score between 1 and 5 based on sufficient evidence.
3. Output Scores in JSON Format: Present the scores in JSON format as follows:
{{
    "completeness": 3,
    "correctness": 4
}}"""
                    
                    messages = [
                        {"role": "user", 
                        "content": [
                                {
                                    "type": "text",
                                    "text": query
                                }
                            ]
                        },
                    ]
                    output = ""
                    response = call_gpt_with_messages(messages, "gpt_4o")
                    output += response.choices[0].message.content

                    # print("query: ", query)
                    # print("answer: ", output + "\n")
                    save_file = os.path.join(f"{DATA_ROOT}/results/MLVU", model, rag_type, video_id, f"{question_id}_sum_score.txt")
                    with open(save_file, 'w') as f:
                        f.write(output)

sub_scene_score()
sum_score()  


# 测试MLVU的model生成准确率

In [ ]:
model = "gpt_4o_20"
rag_type = "gc_rag"
# rag_type = "no_rag"

def sub_scene_score():
    sub_scene_score = []
    sum_score = []
    for root, ds, fs in os.walk(f"{DATA_ROOT}/results/MLVU/{model}/{rag_type}"):
        if len(ds) != 0:
            for video_id in ds:
                preds = os.listdir(os.path.join(root, video_id))
                for pred in preds:
                    question_id = ""
                    if pred.endswith("_sub_score.txt"):
                        question_id = pred.split("_sub_score.txt")[0]
                    elif pred.endswith("_sum_score.txt"):
                        question_id = pred.split("_sum_score.txt")[0]
                    else:
                        continue
                    
                    file_path = os.path.join(root, video_id, pred)
                    with open(file_path, 'r', encoding='utf-8') as pred_file:
                        evaluation = pred_file.read()

                    if pred.endswith("_sub_score.txt"):
                        accuracy = re.findall(r"\"accuracy\"\: ([12345])", evaluation)
                        relevance = re.findall(r"\"relevance\"\: ([12345])", evaluation)
                        if len(accuracy) == 0 or len(relevance) == 0:
                            print(f"{video_id}-{question_id}'s format is not correct!")
                            continue
                        else:
                            avg_score = (int(accuracy[0]) + int(relevance[0])) / 2
                            sub_scene_score.append(avg_score)
                    if pred.endswith("_sum_score.txt"):
                        completeness = re.findall(r"\"completeness\"\: ([12345])", evaluation)
                        correctness = re.findall(r"\"correctness\"\: ([12345])", evaluation)
                        if len(completeness) == 0 or len(correctness) == 0:
                            print(f"{video_id}-{question_id}'s format is not correct!")
                            continue
                        else:
                            avg_score = (int(completeness[0]) + int(correctness[0])) / 2
                            sum_score.append(avg_score)
                    
    print(f"8_sub_scene准确率: {np.mean(sub_scene_score)}")               
    print(f"9_summary准确率: {np.mean(sum_score)}")    
    sub_scene_score.extend(sum_score)
    print(f"平均准确率: {np.mean(sub_scene_score)}")            
sub_scene_score()
